# TAHAP 4 — Model Training (Hybrid: Isolation Forest + Supervised)

Melatih model hybrid dua lapis:
- **Lapis 1:** Isolation Forest (unsupervised) → menghasilkan `if_anomaly_score` sebagai fitur tambahan
- **Lapis 2:** Satu model supervised terbaik (RF vs XGBoost, pilih winner)

**Input:** Splits dari `data/splits/`

**Output:**
- `models/isolation_forest.pkl`
- `models/best_supervised.pkl`
- `models/model_metadata.json`

**Target:** Multi-class classification (LOW=0, MEDIUM=1, HIGH=2, CRITICAL=3)

## Import & Load Splits

In [15]:
import os
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

# XGBoost
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("[!] XGBoost belum terpasang. Jalankan: pip install xgboost")

warnings.filterwarnings('ignore')

# Paths
SPLITS_DIR  = os.path.join("..", "data", "splits")
MODELS_DIR  = os.path.join("..", "models")
RANDOM_STATE = 42

# Load splits
X_train = pd.read_csv(os.path.join(SPLITS_DIR, "X_train.csv"))
X_val   = pd.read_csv(os.path.join(SPLITS_DIR, "X_val.csv"))
X_test  = pd.read_csv(os.path.join(SPLITS_DIR, "X_test.csv"))
y_train = pd.read_csv(os.path.join(SPLITS_DIR, "y_train.csv"))["severity_encoded"]
y_val   = pd.read_csv(os.path.join(SPLITS_DIR, "y_val.csv"))["severity_encoded"]
y_test  = pd.read_csv(os.path.join(SPLITS_DIR, "y_test.csv"))["severity_encoded"]

# Load feature metadata
with open(os.path.join(MODELS_DIR, "feature_columns.json")) as f:
    feat_meta = json.load(f)
SEVERITY_LABELS = feat_meta["severity_labels"]

print(f"[OK] Data loaded")
print(f"     Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"     XGBoost tersedia: {XGBOOST_AVAILABLE}")

[OK] Data loaded
     Train: (2030, 11), Val: (435, 11), Test: (435, 11)
     XGBoost tersedia: True


## Lapis 1 — Isolation Forest (Unsupervised)

Isolation Forest belajar pola normal **tanpa melihat label**. Skor anomali yang dihasilkan menjadi **fitur tambahan** untuk model supervised di Lapis 2.

In [16]:
print("[1/5] Melatih Isolation Forest (unsupervised)...")

# Parameter Isolation Forest
IF_PARAMS = {
    "n_estimators": 200,
    "contamination": 0.10,       # ~10% data dianggap anomali
    "max_features": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

# Pipeline: scaler + IF (scaler terpisah dari scaler utama)
if_pipe = Pipeline([
    ("scaler", RobustScaler()),
    ("iforest", IsolationForest(**IF_PARAMS)),
])

# Train pada data training (tanpa label)
if_pipe.fit(X_train)
print(f"      Training selesai ({IF_PARAMS['n_estimators']} trees)")

# Hitung anomaly score untuk semua splits
# decision_function: makin negatif = makin anomali
# Kita balik tandanya: makin TINGGI = makin aneh
if_score_train = -if_pipe.decision_function(X_train)
if_score_val   = -if_pipe.decision_function(X_val)
if_score_test  = -if_pipe.decision_function(X_test)

# Tambahkan sebagai fitur baru
X_train_aug = X_train.copy()
X_train_aug["if_anomaly_score"] = if_score_train

X_val_aug = X_val.copy()
X_val_aug["if_anomaly_score"] = if_score_val

X_test_aug = X_test.copy()
X_test_aug["if_anomaly_score"] = if_score_test

# Simpan IF model
os.makedirs(MODELS_DIR, exist_ok=True)
IF_PATH = os.path.join(MODELS_DIR, "isolation_forest.pkl")
joblib.dump(if_pipe, IF_PATH)
print(f"      Model IF disimpan: {IF_PATH}")

# Cek deteksi anomali pada training data
preds = if_pipe.predict(X_train)
n_anomaly = (preds == -1).sum()
print(f"      Anomali terdeteksi (train): {n_anomaly} ({n_anomaly/len(X_train)*100:.1f}%)")
print(f"      Fitur sekarang: {X_train_aug.shape[1]} (9 dasar + 1 IF score)")

[1/5] Melatih Isolation Forest (unsupervised)...
      Training selesai (200 trees)
      Model IF disimpan: ..\models\isolation_forest.pkl
      Anomali terdeteksi (train): 203 (10.0%)
      Fitur sekarang: 12 (9 dasar + 1 IF score)


## Lapis 2 — Train Random Forest (Multi-class)

In [17]:
print("[2/5] Melatih Random Forest (multi-class, class_weight=balanced)...")

RF_PARAMS = {
    "n_estimators": 300,
    "max_depth": None,
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

rf_clf = RandomForestClassifier(**RF_PARAMS)
rf_clf.fit(X_train_aug, y_train)

# Evaluasi pada validation set
y_pred_rf = rf_clf.predict(X_val_aug)
rf_f1 = f1_score(y_val, y_pred_rf, average="macro")

print(f"      Training selesai")
print(f"      F1-macro (validation): {rf_f1:.4f}")
print(f"\n  Classification Report (Random Forest, Validation Set):")
print(classification_report(y_val, y_pred_rf,
                            target_names=SEVERITY_LABELS,
                            digits=3, zero_division=0))

[2/5] Melatih Random Forest (multi-class, class_weight=balanced)...
      Training selesai
      F1-macro (validation): 0.9433

  Classification Report (Random Forest, Validation Set):
              precision    recall  f1-score   support

         LOW      0.992     0.984     0.988       375
      MEDIUM      0.871     1.000     0.931        27
        HIGH      0.889     0.889     0.889        18
    CRITICAL      1.000     0.933     0.966        15

    accuracy                          0.979       435
   macro avg      0.938     0.952     0.943       435
weighted avg      0.980     0.979     0.980       435



## Lapis 2 — Train XGBoost (Multi-class)

In [18]:
print("[3/5] Melatih XGBoost (multi-class, softprob)...")

if XGBOOST_AVAILABLE:
    from sklearn.utils.class_weight import compute_sample_weight
    sample_weights = compute_sample_weight("balanced", y_train)

    XGB_PARAMS = {
        "n_estimators": 300,
        "max_depth": 4,
        "min_child_weight": 5,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "gamma": 1,
        "reg_alpha": 0.5,
        "reg_lambda": 2,
        "objective": "multi:softprob",
        "num_class": 4,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "eval_metric": "mlogloss",
    }

    xgb_clf = xgb.XGBClassifier(**XGB_PARAMS)
    xgb_clf.fit(
        X_train_aug, y_train,
        sample_weight=sample_weights,
        eval_set=[(X_val_aug, y_val)],
        verbose=False
    )

    y_pred_xgb = xgb_clf.predict(X_val_aug)
    xgb_f1 = f1_score(y_val, y_pred_xgb, average="macro")

    print(f"      Training selesai")
    print(f"      F1-macro (validation): {xgb_f1:.4f}")
    print(f"\n  Classification Report (XGBoost, Validation Set):")
    print(classification_report(y_val, y_pred_xgb,
                                target_names=SEVERITY_LABELS,
                                digits=3, zero_division=0))
else:
    xgb_clf = None
    xgb_f1 = 0.0
    print("      [!] XGBoost tidak tersedia — lewati")

[3/5] Melatih XGBoost (multi-class, softprob)...
      Training selesai
      F1-macro (validation): 0.9127

  Classification Report (XGBoost, Validation Set):
              precision    recall  f1-score   support

         LOW      0.997     0.960     0.978       375
      MEDIUM      0.818     1.000     0.900        27
        HIGH      0.654     0.944     0.773        18
    CRITICAL      1.000     1.000     1.000        15

    accuracy                          0.963       435
   macro avg      0.867     0.976     0.913       435
weighted avg      0.972     0.963     0.966       435



## Perbandingan & Pilih Model Terbaik

In [19]:
print("[4/5] Membandingkan model pada validation set...")

# Tabel perbandingan
from sklearn.metrics import accuracy_score, precision_score, recall_score

results = {
    "Model": ["Random Forest"],
    "F1-Macro": [rf_f1],
    "Accuracy": [accuracy_score(y_val, y_pred_rf)],
    "Precision-Macro": [precision_score(y_val, y_pred_rf, average="macro", zero_division=0)],
    "Recall-Macro": [recall_score(y_val, y_pred_rf, average="macro", zero_division=0)],
}

if XGBOOST_AVAILABLE and xgb_clf is not None:
    results["Model"].append("XGBoost")
    results["F1-Macro"].append(xgb_f1)
    results["Accuracy"].append(accuracy_score(y_val, y_pred_xgb))
    results["Precision-Macro"].append(precision_score(y_val, y_pred_xgb, average="macro", zero_division=0))
    results["Recall-Macro"].append(recall_score(y_val, y_pred_xgb, average="macro", zero_division=0))

df_results = pd.DataFrame(results)
print("\n  Perbandingan:")
print(df_results.to_string(index=False))

# Pilih pemenang berdasarkan F1-macro
if XGBOOST_AVAILABLE and xgb_f1 > rf_f1:
    best_model = xgb_clf
    best_name = "XGBoost"
    best_type = "xgboost"
    best_f1 = xgb_f1
    best_params = XGB_PARAMS
else:
    best_model = rf_clf
    best_name = "RandomForest"
    best_type = "rf"
    best_f1 = rf_f1
    best_params = RF_PARAMS

print(f"\n  PEMENANG: {best_name} (F1-macro: {best_f1:.4f})")

[4/5] Membandingkan model pada validation set...

  Perbandingan:
        Model  F1-Macro  Accuracy  Precision-Macro  Recall-Macro
Random Forest  0.943348  0.979310         0.937948      0.951556
      XGBoost  0.912747  0.963218         0.867314      0.976111

  PEMENANG: RandomForest (F1-macro: 0.9433)


## Simpan Model Pemenang & Metadata

In [20]:
print("[5/5] Menyimpan model pemenang...")

# Simpan model supervised terbaik
BEST_MODEL_PATH = os.path.join(MODELS_DIR, "best_supervised.pkl")
joblib.dump(best_model, BEST_MODEL_PATH)
print(f"      Model disimpan: {BEST_MODEL_PATH}")

# Simpan metadata model
metadata = {
    "model_name": best_name,
    "model_type": best_type,
    "validation_f1_macro": round(best_f1, 4),
    "num_classes": 4,
    "class_labels": SEVERITY_LABELS,
    "feature_count": X_train_aug.shape[1],
    "feature_columns": list(X_train_aug.columns),
    "training_samples": len(X_train),
    "training_date": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "is_tuned": False,
}

META_PATH = os.path.join(MODELS_DIR, "model_metadata.json")
with open(META_PATH, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"      Metadata disimpan: {META_PATH}")

print(f"\n  Model terpilih: {best_name}")
print(f"  F1-macro (validation): {best_f1:.4f}")
print(f"  Fitur: {X_train_aug.shape[1]} (9 perilaku + 1 IF anomaly score)")
print(f"\nTraining selesai! Lanjut ke Tahap 5 (Evaluation)")

[5/5] Menyimpan model pemenang...
      Model disimpan: ..\models\best_supervised.pkl
      Metadata disimpan: ..\models\model_metadata.json

  Model terpilih: RandomForest
  F1-macro (validation): 0.9433
  Fitur: 12 (9 perilaku + 1 IF anomaly score)

Training selesai! Lanjut ke Tahap 5 (Evaluation)


## Feature Importance (Model Pemenang)

In [21]:
# Tampilkan feature importance dari model pemenang
importances = sorted(
    zip(X_train_aug.columns, best_model.feature_importances_),
    key=lambda t: t[1], reverse=True,
)

print(f"Feature Importance ({best_name}):")
print(f"  {'Fitur':30s} {'Pentingnya':>12s} {'Persentase':>12s}")
total_imp = sum(imp for _, imp in importances)
for name, imp in importances:
    pct = (imp / total_imp * 100) if total_imp > 0 else 0
    print(f"  {name:30s} {imp:12.4f} {pct:11.1f}%")

Feature Importance (RandomForest):
  Fitur                            Pentingnya   Persentase
  refund_count_daily                   0.1638        16.4%
  is_refund                            0.1637        16.4%
  refund_ratio_daily                   0.1179        11.8%
  amount_zscore_cashier                0.0944         9.4%
  amount_rolling_mean_5                0.0913         9.1%
  hour_of_day                          0.0879         8.8%
  if_anomaly_score                     0.0865         8.7%
  time_gap_seconds                     0.0487         4.9%
  txn_freq_daily                       0.0457         4.6%
  amount_deviation_from_mean           0.0345         3.5%
  txn_freq_zscore_cashier              0.0330         3.3%
  is_late_night                        0.0324         3.2%
